# Bonus appendix: Fine-tuned MiniLM

<div dir="rtl">

این نوت‌بوک ضمیمه‌ی bonus_incentivized_reviews.ipynb است. در آنجا مدل‌های کلاسیک روی TF-IDF و بردارهای ثابت MiniLM مقایسه شدند. اینجا خود MiniLM برای تشخیص نظر تشویقی fine-tune می‌شود، یعنی همه‌ی لایه‌هایش با داده‌ی ما آموزش می‌بینند، تا یک مدل DL واقعی هم در جدول مقایسه باشد.

ورودی فایل pool.csv.gz است که نوت‌بوک اصلی ساخته: متن‌های حذف‌شده (masked)، برچسب‌ها و شماره‌ی fold هر نظر. پس داده و foldها دقیقاً همان‌هایی‌اند که بقیه‌ی مدل‌ها با آن سنجیده شدند. این نوت‌بوک به GPU نیاز دارد و روی T4 کولب برای هر fold چند دقیقه طول می‌کشد. خروجی، یعنی AP و AUC هر fold، در minilm_finetune_cv.csv ذخیره می‌شود و نوت‌بوک اصلی آن را به جدول مقایسه اضافه می‌کند. در بخش آخر، مدل یک بار روی کل داده‌ی آموزش آموزش می‌بیند و به نمونه‌ای از نظرهای بعد از ممنوعیت امتیاز می‌دهد تا در نوت‌بوک اصلی روی همان آزمون بعد از ممنوعیت با SVM مقایسه شود.

فایل‌های لازم در پوشه‌ی project2/bonus در Google Drive: pool.csv.gz و post_sample.csv.gz، که هر دو را نوت‌بوک اصلی ساخته است.

</div>

## Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os
import time
import numpy as np
import pandas as pd
import torch
from sklearn.metrics import average_precision_score, roc_auc_score
from transformers import (AutoTokenizer, AutoModelForSequenceClassification, Trainer,
                          TrainingArguments, DataCollatorWithPadding)

out = '/content/drive/MyDrive/project2/bonus/'
print('GPU:', torch.cuda.get_device_name(0))

GPU: Tesla T4


## Data

<div dir="rtl">

چند نظر بعد از masking کاملاً خالی شده‌اند. CSV متن خالی را موقع خواندن NaN می‌کند و tokenizer فقط متن می‌پذیرد، پس دوباره به رشته‌ی خالی تبدیل شدند. این نظرها در نوت‌بوک اصلی هم در داده بودند، پس داده‌ی دو نوت‌بوک یکی می‌ماند.

</div>

In [3]:
pool = pd.read_csv(out + 'pool.csv.gz')
print('empty texts after masking:', pool['masked'].isna().sum())
pool['masked'] = pool['masked'].fillna('')
print('reviews:', len(pool), ' positives:', pool['disclosed'].sum(), f"({pool['disclosed'].mean():.1%})")
pool['fold'].value_counts().sort_index()

empty texts after masking: 55
reviews: 42321  positives: 8388 (19.8%)


,count
fold,
0,8465
1,8464
2,8464
3,8464
4,8464


## Model

<div dir="rtl">

روی MiniLM یک لایه‌ی طبقه‌بندی دوکلاسه اضافه می‌شود و کل مدل با cross-entropy آموزش می‌بیند. حداکثر طول متن ۵۱۲ توکن است، نه ۲۵۶ مثل بردارهای نوت‌بوک اصلی، چون آنجا آخر نزدیک نیمی از نظرهای تشویقی بریده می‌شد و MiniLM تا ۵۱۲ توکن را می‌پذیرد. پارامترها مقادیر رایج fine-tune کردن مدل‌های BERT مانند است: learning rate برابر 5e-5، دو epoch و ۱۰٪ گام‌ها برای warmup. جست‌وجوی پارامتر انجام نشد، چون هر اجرا وقت GPU می‌گیرد.

</div>

In [4]:
name = 'sentence-transformers/all-MiniLM-L6-v2'
tokenizer = AutoTokenizer.from_pretrained(name)

class Reviews(torch.utils.data.Dataset):
    def __init__(self, frame):
        self.enc = tokenizer(frame['masked'].tolist(), truncation=True, max_length=512)
        self.labels = frame['disclosed'].astype(int).tolist()

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, i):
        item = {key: values[i] for key, values in self.enc.items()}
        item['labels'] = self.labels[i]
        return item

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [5]:
def train_model(train_part):
    steps = int(np.ceil(len(train_part) / 32)) * 2
    args = TrainingArguments(output_dir='/content/finetune', num_train_epochs=2, learning_rate=5e-5,
                             per_device_train_batch_size=32, per_device_eval_batch_size=64,
                             warmup_steps=int(0.1 * steps), weight_decay=0.01, fp16=True,
                             save_strategy='no', logging_steps=200, report_to='none', seed=0)
    model = AutoModelForSequenceClassification.from_pretrained(name, num_labels=2)
    trainer = Trainer(model=model, args=args, train_dataset=Reviews(train_part),
                      data_collator=DataCollatorWithPadding(tokenizer))
    trainer.train()
    return trainer

def train_fold(k):
    trainer = train_model(pool[pool['fold'] != k])
    val_part = pool[pool['fold'] == k]
    logits = trainer.predict(Reviews(val_part)).predictions
    return val_part['disclosed'], logits[:, 1] - logits[:, 0]

## Cross-validation

<div dir="rtl">

نتیجه‌ی هر fold بلافاصله در Drive ذخیره می‌شود. اگر اتصال کولب وسط کار قطع شود، اجرای دوباره foldهای تمام‌شده را رد می‌کند.

</div>

In [ ]:
%%time
results_file = out + 'minilm_finetune_cv.csv'
rows = pd.read_csv(results_file).to_dict('records') if os.path.exists(results_file) else []

for k in range(5):
    if k in [row['fold'] for row in rows]:
        continue
    start = time.time()
    labels, scores = train_fold(k)
    rows.append({'fold': k, 'AP': average_precision_score(labels, scores), 'AUC': roc_auc_score(labels, scores),
                 'minutes': (time.time() - start) / 60})
    pd.DataFrame(rows).to_csv(results_file, index=False)
    print(rows[-1])

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.504413
400,0.398166
600,0.360489
800,0.356135
1000,0.349807
1200,0.317602
1400,0.312094
1600,0.303731
1800,0.303369
2000,0.303722


{'fold': 0, 'AP': np.float64(0.6261444169882076), 'AUC': np.float64(0.8707498953960344), 'minutes': 7.531600467363993}


... (12 outputs left out)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.505948
400,0.401613
600,0.372689
800,0.351295
1000,0.348038
1200,0.330782
1400,0.309458
1600,0.316714
1800,0.307256
2000,0.298980


{'fold': 4, 'AP': np.float64(0.6504746590952467), 'AUC': np.float64(0.881125235602828), 'minutes': 7.355185500780741}
CPU times: user 37min 27s, sys: 9.2 s, total: 37min 36s
Wall time: 37min 6s


In [ ]:
folds = pd.read_csv(results_file)
display(folds.round(3))
print('AP:', round(folds['AP'].mean(), 3), '+/-', round(folds['AP'].std(ddof=0), 3))
print('AUC:', round(folds['AUC'].mean(), 3))

,fold,AP,AUC,minutes
0,0,0.626,0.871,7.532
1,1,0.626,0.877,7.428
2,2,0.660,0.885,7.432
3,3,0.616,0.874,7.365
4,4,0.650,0.881,7.355


AP: 0.636 +/- 0.016
AUC: 0.878


## After the ban

<div dir="rtl">

مدل یک بار روی کل داده‌ی آموزش، با همان تنظیم‌ها، آموزش می‌بیند و به نمونه‌ای از نظرهای بعد از ممنوعیت امتیاز می‌دهد. این نمونه (post_sample.csv.gz) را نوت‌بوک اصلی ساخته است: همه‌ی نظرهای نظردهنده‌های کنارگذاشته (held out)، نظرهای اعضای حلقه‌ها و ۵۰ هزار نظر تصادفی دیگر. امتیازها در post_sample_finetune.csv ذخیره می‌شوند و مقایسه‌ی کامل با SVM در نوت‌بوک اصلی انجام می‌شود.

</div>

In [6]:
%%time
sample = pd.read_csv(out + 'post_sample.csv.gz')
sample['masked'] = sample['masked'].fillna('')

trainer = train_model(pool)
logits = trainer.predict(Reviews(sample.assign(disclosed=False))).predictions
sample['finetune_score'] = logits[:, 1] - logits[:, 0]
sample[['row', 'finetune_score']].to_csv(out + 'post_sample_finetune.csv', index=False)
print('scored reviews:', len(sample))

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key               | Status  | 
------------------+---------+-
classifier.weight | MISSING | 
classifier.bias   | MISSING | 

Notes:
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.488595
400,0.393403
600,0.373028
800,0.364366
1000,0.352509
1200,0.334092
1400,0.331028
1600,0.313300
1800,0.307678
2000,0.288894


scored reviews: 53648
CPU times: user 10min 54s, sys: 4.45 s, total: 10min 59s
Wall time: 10min 51s


In [7]:
test = sample[~sample['ring']]
print('held out vs others, fine-tuned AUC:', round(roc_auc_score(test['group'] == 'held out', test['finetune_score']), 3))
print('held out vs others, SVM AUC:', round(roc_auc_score(test['group'] == 'held out', test['svm_score']), 3))

held out vs others, fine-tuned AUC: 0.722
held out vs others, SVM AUC: 0.716
